**Student Name:** Shelia Howard  
**Course:** DSC670-T301 Advanced Uses of Generative AI (2271-1)  
**Assignment:** Term Project - Milestone 2  
**Due Date:** 10/04/2026  
**Dataset(s):** `healthcare_mapping_prompt_experiments.csv` - synthetic healthcare demographic mapping examples created for this milestone; no PHI is used.

# DSC670 Term Project - Milestone 2: Healthcare Data Mapping Assistant

## Introduction

This notebook supports Milestone 2 of the Healthcare Data Mapping Assistant term project. It establishes a controlled healthcare demographic target schema, loads a small synthetic evaluation dataset, defines six prompt-engineering experiments, calls a base OpenAI language model, and organizes the responses for comparison. The purpose is to determine whether prompt design can produce accurate, consistent, and review-aware source-to-target mapping recommendations before the project moves to supervised fine-tuning.

The notebook deliberately includes both straightforward mappings and ambiguous cases. A useful assistant should not only identify obvious semantic matches; it should also recognize when the available information does not justify a safe automatic mapping. Those cases are expected to return `REVIEW` and remain in the human analyst's decision process.

Throughout the notebook, each code block includes comments explaining what the code is doing and why the step matters.


## 1. Import Required Libraries


In [1]:
import os
import json
import re
import pandas as pd
from openai import OpenAI

### Why This Code Matters

The imports support the full experiment workflow rather than only the API call. Environment variables protect the API key, pandas provides reproducible tabular evaluation, and JSON and regular expressions allow outputs from different prompt styles to be inspected consistently. These tools make it possible to compare prompts using the same cases instead of relying on informal impressions.


## 2. Configure the Model and API Client


In [2]:
# Import getpass so an API key can be entered securely without displaying or saving the key in the notebook.
from getpass import getpass

# Read the API key from the OPENAI_API_KEY environment variable when one has already been configured.
api_key = os.getenv("OPENAI_API_KEY")

# Check whether an environment variable supplied the API key.
if not api_key:
    # Prompt for the API key securely when the environment variable is unavailable.
    api_key = getpass("Enter the OpenAI API key: ")

# Create an authenticated OpenAI client that will be reused for all prompt experiments.
client = OpenAI(api_key=api_key)

# Store the baseline model name in one variable so the experiment uses the same model consistently.
MODEL = "gpt-4o-mini"

# Display the model identifier without displaying the API credential.
print(f"Baseline model selected: {MODEL}")

Enter the OpenAI API key:  ········


Baseline model selected: gpt-4o-mini


### Why This Code Matters

Keeping the model identifier in one variable improves reproducibility and makes the notebook adaptable if the course requires a different OpenAI model. The API key is deliberately excluded from the notebook so that the file can be submitted or stored without exposing a credential.


## 3. Define the Controlled Target Schema


In [3]:
# Create the only target field names that the mapping assistant is allowed to recommend.
target_schema = [
    "member_id",
    "first_name",
    "middle_name",
    "last_name",
    "date_of_birth",
    "sex",
    "address_line_1",
    "address_line_2",
    "city",
    "state",
    "postal_code",
    "phone_number",
    "email_address",
]
# Join the schema into one comma-separated string so the same vocabulary can be inserted into multiple prompts.
TARGET_SCHEMA_TEXT = ", ".join(target_schema)
# Print the schema so the notebook documents the exact target vocabulary used in the experiments.
print(TARGET_SCHEMA_TEXT)


member_id, first_name, middle_name, last_name, date_of_birth, sex, address_line_1, address_line_2, city, state, postal_code, phone_number, email_address


### Why This Code Matters

A controlled target schema converts an open-ended generation problem into a constrained recommendation task. This is important for healthcare data mapping because a plausible but invented target field would still be unusable. The same schema will later support scoring and the final application interface.


## 4. Load the Synthetic Evaluation Dataset


In [4]:
# Store the relative CSV path so the notebook can be run from the folder containing the submitted files.
DATA_PATH = "healthcare_mapping_prompt_experiments.csv"
# Read the synthetic healthcare mapping cases into a pandas DataFrame.
df = pd.read_csv(DATA_PATH)
# Display the first ten records so the source fields, expected targets, and review labels can be inspected.
display(df.head(10))
# Display the number of rows so the scope of this Milestone 2 evaluation set is explicit.
print(f"Synthetic evaluation rows: {len(df)}")


,source_field,datatype,description,example_value,expected_target,human_review_expected
0,patientDOB,date,Patient date of birth,1988-04-11,date_of_birth,NO
1,member_birth_date,date,Member birth date,1975-09-30,date_of_birth,NO
2,birth_dt,date,Date member was born,2001-01-05,date_of_birth,NO
3,mem_num,string,Unique health plan member identifier,M00044321,member_id,NO
4,subscriber_id,string,Identifier assigned to member/subscriber,S893440,member_id,NO
5,fname,string,Member first or given name,Jordan,first_name,NO
6,given_name,string,Patient given name,Alex,first_name,NO
7,mname,string,Member middle name,Lee,middle_name,NO
8,surname,string,Patient family name,Howard,last_name,NO
9,lname,string,Member last name,Garcia,last_name,NO


Synthetic evaluation rows: 30


### Why This Code Matters

The dataset contains synthetic examples only and does not include protected health information. It includes common naming variations and deliberately ambiguous cases. The expected target and human-review labels provide a reference for comparing model behavior, while the small size keeps the Milestone 2 experiment manageable. This file is an evaluation aid, not the final 200-300 example fine-tuning dataset.


## 5. Select Reusable Test Cases


In [5]:
# Select a small mix of straightforward and ambiguous records so every prompt is tested on the same evidence.
test_fields = ["patientDOB", "mem_num", "addr2", "id", "name", "contact"]
# Filter the full synthetic dataset to only the reusable test cases listed above.
test_cases = df[df["source_field"].isin(test_fields)].copy()
# Reset the row index so the experiment table is clean and sequential.
test_cases.reset_index(drop=True, inplace=True)
# Display the reusable cases before any model calls are made.
display(test_cases)


,source_field,datatype,description,example_value,expected_target,human_review_expected
0,patientDOB,date,Patient date of birth,1988-04-11,date_of_birth,NO
1,mem_num,string,Unique health plan member identifier,M00044321,member_id,NO
2,addr2,string,"Apartment, suite, or secondary address informa...",Apt 4B,address_line_2,NO
3,id,string,Unique identifier,12345,REVIEW,YES
4,name,string,Full legal name in one field,Jordan Lee Howard,REVIEW,YES
5,contact,string,Preferred contact value; type not supplied,6155550199,REVIEW,YES


### Why This Code Matters

I selected the same six cases for every prompt so I could compare the prompts fairly. I intentionally chose three direct mappings and three ambiguous cases because I wanted to test both mapping accuracy and whether the model would recognize when it should defer to a human reviewer. Three cases have direct mappings and three require careful uncertainty handling. This allows the experiment to test both semantic mapping capability and the model's willingness to avoid an unsupported answer.

## 6. Define Six Prompt Experiments


In [6]:
# Create a dictionary so each experiment has a stable name and a reusable prompt template.
prompt_templates = {
# Experiment 1 provides a minimal baseline with no controlled target vocabulary.
    "P1_Minimal": """Map the following healthcare source field to the best target field. Source field: {source_field}. Datatype: {datatype}. Description: {description}. Example value: {example_value}. Return the target field and a short reason.""",
# Experiment 2 restricts the answer to the approved schema.
    "P2_ControlledSchema": """You are mapping healthcare demographic data. Choose the best target only from this approved schema: {target_schema}. Source field: {source_field}. Datatype: {datatype}. Description: {description}. Example value: {example_value}. Return the selected target and one sentence explaining the match.""",
# Experiment 3 adds an analyst role and tells the model how to weigh the available evidence.
    "P3_RoleAndCriteria": """Act as a healthcare interoperability analyst. Map the source field to exactly one approved target field from this list: {target_schema}. Evaluate the field name, datatype, description, and example value together. Prefer semantic meaning over superficial name similarity. Source field: {source_field}. Datatype: {datatype}. Description: {description}. Example value: {example_value}. Return Target: <field> and Reason: <brief explanation>.""",
# Experiment 4 explicitly allows REVIEW when a safe mapping is not supported.
    "P4_HumanReview": """Act as a healthcare interoperability analyst. Choose one target from the approved schema or choose REVIEW when the information is ambiguous, contradictory, or cannot be represented by one target field. Approved schema: {target_schema}. Source field: {source_field}. Datatype: {datatype}. Description: {description}. Example value: {example_value}. Return Target: <field or REVIEW>, Human review: <YES or NO>, and Reason: <brief explanation>.""",
# Experiment 5 provides one straightforward example and one abstention example before the new case.
    "P5_FewShot": """Act as a healthcare interoperability analyst. Choose one target from the approved schema or REVIEW if the mapping is not sufficiently supported. Approved schema: {target_schema}. Example 1: source=patientDOB; datatype=date; description=Patient date of birth; output=date_of_birth, review=NO. Example 2: source=name; datatype=string; description=Full legal name in one field; output=REVIEW, review=YES. Now evaluate: Source field: {source_field}. Datatype: {datatype}. Description: {description}. Example value: {example_value}. Return Target: <field or REVIEW>, Human review: <YES or NO>, and Reason: <brief explanation>.""",
# Experiment 6 requests a machine-readable response for later application integration.
    "P6_JSON": """Act as a healthcare interoperability analyst. Select only an approved target field or REVIEW when evidence is insufficient. Approved schema: {target_schema}. Consider field name, datatype, description, and example value. Source field: {source_field}. Datatype: {datatype}. Description: {description}. Example value: {example_value}. Return only valid JSON with exactly these keys: \"target_field\", \"human_review\", \"reason\". Set \"human_review\" to true when the answer is REVIEW or when material ambiguity remains. Do not add fields outside the approved schema.""",
}
# Display the experiment names so the notebook documents all prompt variants before execution.
print(list(prompt_templates.keys()))


['P1_Minimal', 'P2_ControlledSchema', 'P3_RoleAndCriteria', 'P4_HumanReview', 'P5_FewShot', 'P6_JSON']


### Why This Code Matters

The six prompts form a deliberate progression. The experiments test whether target-schema grounding, professional role context, explicit decision criteria, an abstention rule, few-shot demonstrations, and a structured output contract each improve behavior. This directly addresses the Milestone 2 requirement to experiment with prompts before fine-tuning.


## 7. Create a Reusable API Function


In [7]:
# Define one function so every experiment is sent to the model in the same way.
def ask_model(prompt):
    # Send the completed prompt to the selected baseline model using the Responses API.
    response = client.responses.create(model=MODEL, input=prompt, temperature=0)
    # Return only the model's combined text output so it can be stored directly in the results table.
    return response.output_text.strip()


### Why This Code Matters

A single API function keeps model settings constant across the prompt experiments. Temperature is set to zero to reduce avoidable randomness during comparison. The experiment is intended to measure the effect of prompt design, so changing sampling behavior from one prompt to another would make interpretation more difficult.


## 8. Run Every Prompt Against the Same Test Cases


In [8]:
# Create an empty list that will collect one result record for every prompt-and-case combination.
results = []
# Loop through each named prompt template so every experiment is executed.
for prompt_name, template in prompt_templates.items():
    # Loop through each selected source-field case so all prompts see the same test inputs.
    for _, row in test_cases.iterrows():
        # Fill the current prompt template with the approved schema and the current synthetic source-field details.
        prompt = template.format(target_schema=TARGET_SCHEMA_TEXT, source_field=row["source_field"], datatype=row["datatype"], description=row["description"], example_value=row["example_value"])
        # Send the completed prompt to the OpenAI model and capture the generated answer.
        output = ask_model(prompt)
        # Add the experiment name, expected answer, and actual model output to the results collection.
        results.append({"prompt": prompt_name, "source_field": row["source_field"], "expected_target": row["expected_target"], "expected_review": row["human_review_expected"], "model_output": output})
# Convert the collected list of dictionaries into a DataFrame for easier comparison.
results_df = pd.DataFrame(results)
# Display the complete experiment result table.
display(results_df)


,prompt,source_field,expected_target,expected_review,model_output
0,P1_Minimal,patientDOB,date_of_birth,NO,**Target Field:** patientBirthDate\n\n**Reason...
1,P1_Minimal,mem_num,member_id,NO,**Target Field: member_id**\n\n**Reason:** The...
2,P1_Minimal,addr2,address_line_2,NO,**Target Field: Secondary Address Line**\n\n**...
3,P1_Minimal,id,REVIEW,YES,**Target Field: patient_id**\n\n**Reason:** Th...
4,P1_Minimal,name,REVIEW,YES,**Target Field: Patient Name**\n\n**Reason:** ...
5,P1_Minimal,contact,REVIEW,YES,**Target Field:** `phone_number`\n\n**Reason:*...
6,P2_ControlledSchema,patientDOB,date_of_birth,NO,The best target is **date_of_birth**. This mat...
7,P2_ControlledSchema,mem_num,member_id,NO,**Target:** member_id\n\n**Explanation:** The ...
8,P2_ControlledSchema,addr2,address_line_2,NO,**Target:** address_line_2\n\n**Explanation:**...
9,P2_ControlledSchema,id,REVIEW,YES,**Target:** member_id\n\n**Explanation:** The ...


### Why This Code Matters

This cell produces the actual Milestone 2 prompt experiment evidence. Every prompt is evaluated on identical cases, and each generated response is stored beside the expected result. This makes the comparison reproducible and provides material that can be discussed in the paper without relying on memory or isolated screenshots.


## 9. Evaluate the Prompt Experiment Results

This section evaluates the model responses using the same four dimensions described in the Milestone 2 paper: target-field accuracy, controlled-vocabulary adherence, human-review behavior, and response-format consistency.

The evaluation first parses the model's target recommendation and human-review decision from each response. This is important because searching for words anywhere in the response could produce misleading results. For example, a response could recommend `member_id` while also containing the phrase "human review," which should not be interpreted as a `REVIEW` target.

The evaluation also applies different format checks based on the instructions given to each prompt. Earlier prompts intentionally allow less structured output, while the later prompts require labeled fields or valid JSON.

In [9]:
# Import the JSON module so the structured Prompt 6 responses can be tested for valid JSON.
import json

# Import the regular expression module so labeled values can be extracted from text responses.
import re

# Define a helper function that removes Markdown code fences from JSON-like responses.
def clean_json_text(model_output):
    # Convert the model response to text and remove leading and trailing whitespace.
    text = str(model_output).strip()

    # Remove an opening ```json or ``` Markdown code fence when one is present.
    text = re.sub(
        r"^```(?:json)?\s*",
        "",
        text,
        flags=re.IGNORECASE,
    )

    # Remove a closing ``` Markdown code fence when one is present.
    text = re.sub(
        r"\s*```$",
        "",
        text,
    )

    # Return the cleaned content so it can be parsed as JSON.
    return text.strip()

# Define a function that extracts the model's target-field decision from a response.
def extract_target(model_output):
    # Convert the model output to cleaned text and remove Markdown code fences when present.
    text = clean_json_text(model_output)

    # Try to interpret the cleaned response as JSON because Prompt 6 requests JSON output.
    try:
        # Convert valid JSON text into a Python dictionary.
        parsed_json = json.loads(text)

        # Check whether the JSON response contains the expected target_field key.
        if isinstance(parsed_json, dict) and "target_field" in parsed_json:
            # Return the target-field value as cleaned text.
            return str(parsed_json["target_field"]).strip()

    # Continue with text-based parsing when the response is not valid JSON.
    except (json.JSONDecodeError, TypeError):
        pass

    # Search for responses that explicitly use a label such as "Target: date_of_birth".
    labeled_target = re.search(
        r"Target\s*:\s*([A-Za-z0-9_]+)",
        text,
        flags=re.IGNORECASE,
    )

    # Return the labeled target value when a Target label is present.
    if labeled_target:
        return labeled_target.group(1).strip()

    # Check whether the response begins with REVIEW as the actual mapping decision.
    if re.search(r"^\s*REVIEW\b", text, flags=re.IGNORECASE):
        # Return REVIEW as the parsed target decision.
        return "REVIEW"

    # Search the complete response for one of the approved target-schema field names.
    for field in target_schema:
        # Use token boundaries so a target name is not matched as part of a longer unrelated string.
        if re.search(rf"\b{re.escape(field)}\b", text, flags=re.IGNORECASE):
            # Return the approved target field that was found.
            return field

    # Examine the first nonempty line as a fallback for early prompts that may return only a field name.
    first_line = next(
        (line.strip() for line in text.splitlines() if line.strip()),
        "",
    )

    # Remove common introductory labels from the first line.
    first_line = re.sub(
        r"^(Target|Selected target|Best target|Mapping)\s*:\s*",
        "",
        first_line,
        flags=re.IGNORECASE,
    )

    # Extract the first simple field-like token from the cleaned first line.
    fallback_match = re.match(r"([A-Za-z][A-Za-z0-9_]*)", first_line)

    # Return the fallback target when a field-like value can be identified.
    if fallback_match:
        return fallback_match.group(1).strip()

    # Return None when the response does not contain a target decision that can be reliably identified.
    return None


# Define a function that determines whether the model requested human review.
def extract_review_decision(model_output, parsed_target):
    # Convert the response to cleaned text and remove Markdown code fences when present.
    text = clean_json_text(model_output)

    # Try JSON parsing first because Prompt 6 uses a boolean human_review field.
    try:
        # Convert valid JSON into a Python dictionary.
        parsed_json = json.loads(text)

        # Check whether the expected human_review key is available.
        if isinstance(parsed_json, dict) and "human_review" in parsed_json:
            # Store the value from the JSON response.
            review_value = parsed_json["human_review"]

            # Return the value directly when the model supplied a true Python Boolean.
            if isinstance(review_value, bool):
                return review_value

            # Convert text versions such as "true", "yes", or "1" into True.
            return str(review_value).strip().lower() in {"true", "yes", "1"}

    # Continue with text parsing when the output is not valid JSON.
    except (json.JSONDecodeError, TypeError):
        pass

    # Search for an explicit Human review label in the labeled prompt responses.
    labeled_review = re.search(
        r"Human\s*review\s*:\s*(YES|NO|TRUE|FALSE)",
        text,
        flags=re.IGNORECASE,
    )

    # Convert an explicitly labeled review response into a Boolean value.
    if labeled_review:
        return labeled_review.group(1).upper() in {"YES", "TRUE"}

    # Treat REVIEW as requiring human review when it is the model's parsed target decision.
    if str(parsed_target).strip().upper() == "REVIEW":
        return True

    # Assume no review request when the response contains no explicit review decision.
    return False


# Define a function that checks whether the structured JSON prompt returned valid JSON.
def check_json_validity(prompt, model_output):
    # Return a missing value for prompts that were not instructed to produce JSON.
    if prompt != "P6_JSON":
        return None

    # Convert the model response to cleaned text.
    text = str(model_output).strip()

    # Try to parse the response as JSON.
    try:
        # Convert the response into a Python object.
        parsed_json = json.loads(text)

    # Return False when JSON parsing fails.
    except (json.JSONDecodeError, TypeError):
        return False

    # Confirm that the parsed JSON response is a dictionary.
    if not isinstance(parsed_json, dict):
        return False

    # Define the exact keys required by the structured JSON prompt.
    required_keys = {"target_field", "human_review", "reason"}

    # Return True only when the response contains exactly the required keys.
    return set(parsed_json.keys()) == required_keys


# Define a function that checks whether each response follows the format requested by its prompt.
def check_format_validity(prompt, model_output):
    # Convert the response to cleaned text.
    text = str(model_output).strip()

    # An empty response does not satisfy any prompt format.
    if not text:
        return False

    # The minimal prompt intentionally does not require a labeled structure.
    if prompt == "P1_Minimal":
        # Treat a nonempty response as format compliant.
        return True

    # The controlled-schema prompt requests a target and explanation but no fixed labels.
    if prompt == "P2_ControlledSchema":
        # Treat a nonempty response as format compliant.
        return True

    # The role-and-criteria prompt explicitly requests Target and Reason labels.
    if prompt == "P3_RoleAndCriteria":
        # Confirm that both requested labels appear.
        return bool(
            re.search(r"Target\s*:", text, flags=re.IGNORECASE)
            and re.search(r"Reason\s*:", text, flags=re.IGNORECASE)
        )

    # The human-review and few-shot prompts request Target, Human review, and Reason labels.
    if prompt in {"P4_HumanReview", "P5_FewShot"}:
        # Confirm that all three requested labels appear.
        return bool(
            re.search(r"Target\s*:", text, flags=re.IGNORECASE)
            and re.search(r"Human\s*review\s*:", text, flags=re.IGNORECASE)
            and re.search(r"Reason\s*:", text, flags=re.IGNORECASE)
        )

    # The final prompt requests valid JSON with exactly three specified keys.
    if prompt == "P6_JSON":
        # Reuse the JSON-validity function for the structured response.
        return check_json_validity(prompt, text) is True

    # Return False if an unexpected prompt identifier appears.
    return False


# Extract the model's target decision from every response.
results_df["parsed_target"] = results_df["model_output"].apply(extract_target)


# Extract the model's human-review decision from every response.
results_df["predicted_review"] = results_df.apply(
    lambda row: extract_review_decision(
        row["model_output"],
        row["parsed_target"],
    ),
    axis=1,
)


# Convert the expected human-review value from YES/NO text into a Boolean value.
results_df["expected_review_bool"] = (
    results_df["expected_review"]
    .astype(str)
    .str.strip()
    .str.upper()
    .eq("YES")
)


# Compare the parsed target with the expected target using case-insensitive exact matching.
results_df["target_correct"] = (
    results_df["parsed_target"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
    ==
    results_df["expected_target"]
    .astype(str)
    .str.strip()
    .str.lower()
)


# Create the complete set of valid mapping decisions, including REVIEW.
valid_targets = set(target_schema + ["REVIEW"])


# Check whether each parsed model decision remains inside the approved target vocabulary.
results_df["schema_adherent"] = results_df["parsed_target"].isin(valid_targets)


# Compare the model's review decision with the expected review decision.
results_df["review_correct"] = (
    results_df["predicted_review"]
    ==
    results_df["expected_review_bool"]
)


# Identify ambiguous cases that were correctly escalated to human review.
results_df["ambiguity_detected"] = (
    results_df["expected_review_bool"]
    &
    results_df["predicted_review"]
)


# Identify unnecessary review requests on cases that should have been mapped without escalation.
results_df["unnecessary_review"] = (
    ~results_df["expected_review_bool"]
    &
    results_df["predicted_review"]
)


# Check JSON validity specifically for the structured JSON prompt.
results_df["json_valid"] = results_df.apply(
    lambda row: check_json_validity(
        row["prompt"],
        row["model_output"],
    ),
    axis=1,
)


# Check whether each response follows the output format requested by its prompt.
results_df["format_valid"] = results_df.apply(
    lambda row: check_format_validity(
        row["prompt"],
        row["model_output"],
    ),
    axis=1,
)


# Display the detailed evaluation results so individual successes and failures can be inspected.
display(
    results_df[
        [
            "prompt",
            "source_field",
            "expected_target",
            "parsed_target",
            "target_correct",
            "schema_adherent",
            "expected_review",
            "predicted_review",
            "review_correct",
            "unnecessary_review",
            "format_valid",
            "json_valid",
        ]
    ]
)

,prompt,source_field,expected_target,parsed_target,target_correct,schema_adherent,expected_review,predicted_review,review_correct,unnecessary_review,format_valid,json_valid
0,P1_Minimal,patientDOB,date_of_birth,NaN,False,False,NO,False,True,False,True,None
1,P1_Minimal,mem_num,member_id,member_id,True,True,NO,False,True,False,True,None
2,P1_Minimal,addr2,address_line_2,NaN,False,False,NO,False,True,False,True,None
3,P1_Minimal,id,REVIEW,NaN,False,False,YES,False,False,False,True,None
4,P1_Minimal,name,REVIEW,NaN,False,False,YES,False,False,False,True,None
5,P1_Minimal,contact,REVIEW,phone_number,False,True,YES,False,False,False,True,None
6,P2_ControlledSchema,patientDOB,date_of_birth,date_of_birth,True,True,NO,False,True,False,True,None
7,P2_ControlledSchema,mem_num,member_id,member_id,True,True,NO,False,True,False,True,None
8,P2_ControlledSchema,addr2,address_line_2,address_line_2,True,True,NO,False,True,False,True,None
9,P2_ControlledSchema,id,REVIEW,member_id,False,True,YES,False,False,False,True,None


### Evaluation Commentary

The evaluation separates the model's target decision from other language that may appear in the response. This distinction is especially important for ambiguous cases because the word `REVIEW` may appear in a rationale or human-review statement without representing the model's actual mapping decision. Target accuracy therefore compares the parsed mapping decision directly with the expected target rather than searching the complete response for matching words.

The `target_correct` measure evaluates whether the model selected the expected target field or appropriately selected `REVIEW`. The `schema_adherent` measure determines whether the parsed target remained within the approved target vocabulary or used the permitted `REVIEW` outcome. The `review_correct` measure evaluates whether the model's human-review decision agrees with the expected review label. This allows the evaluation to reward appropriate escalation of ambiguous cases while also identifying unnecessary escalation of straightforward mappings.

Response-format consistency is evaluated separately from mapping correctness. Prompts 1 and 2 intentionally allow relatively flexible text responses, while Prompts 3 through 5 request increasingly specific labels. Prompt 6 is evaluated more strictly because it requests only valid JSON. This distinction is important because a model can make the correct mapping decision while still failing to follow the requested response format. For an application-oriented use case, both the semantic decision and the response structure matter.

In [10]:
# Define a helper function that calculates the review sensitivity for one prompt's results.
def calculate_review_sensitivity(group):
    # Keep only cases that were intentionally designed to require human review.
    review_cases = group[group["expected_review_bool"]]

    # Return a missing value if a prompt group contains no review-required examples.
    if len(review_cases) == 0:
        return None

    # Calculate the proportion of ambiguous cases that the model correctly sent for review.
    return review_cases["predicted_review"].mean()


# Define a helper function that calculates the unnecessary-review rate for one prompt.
def calculate_unnecessary_review_rate(group):
    # Keep only cases that should not require human review.
    non_review_cases = group[~group["expected_review_bool"]]

    # Return a missing value if a prompt group contains no straightforward examples.
    if len(non_review_cases) == 0:
        return None

    # Calculate the proportion of straightforward mappings that were unnecessarily escalated.
    return non_review_cases["predicted_review"].mean()


# Create an empty list that will store one summary record for each prompt experiment.
summary_rows = []


# Group the detailed experiment results by prompt so each prompt can be evaluated separately.
for prompt, group in results_df.groupby("prompt", sort=False):
    # Create one dictionary containing the main evaluation measures for the current prompt.
    summary_row = {
        # Store the prompt identifier.
        "prompt": prompt,

        # Calculate the percentage of cases with the correct target-field decision.
        "target_accuracy": group["target_correct"].mean(),

        # Calculate the percentage of responses that stayed inside the approved decision vocabulary.
        "schema_adherence": group["schema_adherent"].mean(),

        # Calculate the percentage of ambiguous cases that were correctly escalated.
        "review_sensitivity": calculate_review_sensitivity(group),

        # Calculate how often straightforward cases were unnecessarily sent for review.
        "unnecessary_review_rate": calculate_unnecessary_review_rate(group),

        # Calculate overall agreement between expected and predicted review decisions.
        "review_accuracy": group["review_correct"].mean(),

        # Calculate the percentage of responses that followed the format requested by the prompt.
        "format_consistency": group["format_valid"].mean(),
    }

    # Add the Prompt 6 JSON-validity score only when the current group is Prompt 6.
    if prompt == "P6_JSON":
        # Calculate the percentage of Prompt 6 responses that were valid JSON with exactly the required keys.
        summary_row["json_validity"] = group["json_valid"].fillna(False).mean()

    # Use a missing value for JSON validity when the prompt did not request JSON.
    else:
        summary_row["json_validity"] = None

    # Add the completed summary record to the list.
    summary_rows.append(summary_row)


# Convert the prompt-level summary records into a DataFrame.
prompt_summary = pd.DataFrame(summary_rows)


# Create a copy that will be formatted as percentages for easier interpretation.
prompt_summary_display = prompt_summary.copy()


# Identify the metric columns that should be displayed as percentages.
percentage_columns = [
    "target_accuracy",
    "schema_adherence",
    "review_sensitivity",
    "unnecessary_review_rate",
    "review_accuracy",
    "format_consistency",
    "json_validity",
]


# Loop through each metric column.
for column in percentage_columns:
    # Convert numeric proportions such as 0.8333 into readable percentage strings such as 83.3%.
    prompt_summary_display[column] = prompt_summary_display[column].apply(
        lambda value: f"{value:.1%}" if pd.notna(value) else "N/A"
    )


# Display the prompt-level summary table for comparison across all six experiments.
display(prompt_summary_display)

,prompt,target_accuracy,schema_adherence,review_sensitivity,unnecessary_review_rate,review_accuracy,format_consistency,json_validity
0,P1_Minimal,16.7%,33.3%,0.0%,0.0%,50.0%,100.0%,N/A
1,P2_ControlledSchema,50.0%,100.0%,0.0%,0.0%,50.0%,100.0%,N/A
2,P3_RoleAndCriteria,50.0%,100.0%,0.0%,0.0%,50.0%,100.0%,N/A
3,P4_HumanReview,83.3%,100.0%,66.7%,0.0%,83.3%,100.0%,N/A
4,P5_FewShot,83.3%,100.0%,66.7%,0.0%,83.3%,100.0%,N/A
5,P6_JSON,83.3%,100.0%,100.0%,33.3%,83.3%,0.0%,0.0%


### Prompt-Level Results
The results show a clear difference between prompts that only constrain the target vocabulary and prompts that explicitly allow the model to defer an uncertain mapping for human review. The minimal prompt, P1_Minimal, achieved 16.7% target accuracy and 33.3% schema adherence. Without an approved target vocabulary, the model generated several plausible but unusable alternatives, including `patientBirthDate`, `patient_id`, and descriptive field names that were not part of the controlled schema. The higher schema-adherence score relative to target accuracy occurred because some responses happened to contain an approved target even when the overall mapping decision did not match the expected result. This illustrates why schema adherence and target accuracy measure different aspects of model behavior.

P2_ControlledSchema and P3_RoleAndCriteria both improved target accuracy to 50.0% and schema adherence to 100.0%. Both prompts correctly mapped all three straightforward cases. However, neither prompt allowed `REVIEW` as a valid outcome, so all three ambiguous cases were forced into specific target fields. Adding the analyst role and additional reasoning criteria in P3 did not improve the measured results over P2 for this six-case evaluation set. This suggests that additional role context alone was not enough to solve the uncertainty-handling problem.

The largest measured improvement occurred when an explicit human-review option was introduced. P4_HumanReview achieved 83.3% target accuracy and correctly escalated two of the three ambiguous cases, producing a review sensitivity of 66.7%. It correctly returned `REVIEW` for the generic `id` field and the combined `name` field but mapped the ambiguous `contact` field to `phone_number`. P5_FewShot also achieved 83.3% target accuracy and 66.7% review sensitivity. In that experiment, the model correctly escalated `name` and `contact` but mapped the ambiguous `id` field to `member_id`. The few-shot examples therefore did not improve the aggregate score over the explicit review rule, although the specific error pattern changed.

The unnecessary-review rate was 0.0% for Prompts 1 through 5, meaning none of the three straightforward test cases was unnecessarily escalated by those prompts. This is important because high review sensitivity would be less useful if it were achieved simply by sending every field to a human analyst. Taken together, the results indicate that explicit uncertainty handling contributed more to the improvement in this experiment than role instructions or schema constraints alone.

Prompt 6 produced a different pattern. After removing the Markdown code fences for semantic evaluation, P6_JSON achieved 83.3% target accuracy, 100.0% schema adherence, and 100.0% review sensitivity. It correctly identified all three ambiguous cases as requiring review. However, it unnecessarily escalated one of the three straightforward cases, `mem_num`, which should have mapped to `member_id`. This produced an unnecessary-review rate of 33.3% and an overall review accuracy of 83.3%.

P6_JSON also received 0.0% format consistency and 0.0% strict JSON validity because every response was wrapped inside Markdown code fences even though the prompt explicitly requested only valid JSON. The underlying JSON content could be parsed after preprocessing, but the original responses did not satisfy the exact output-format requirement. This distinction demonstrates that semantic mapping performance and application-ready response formatting must be evaluated separately.

In [13]:
# Create a simplified results table containing the four primary evaluation dimensions described in the paper.
paper_results_table = prompt_summary[
    [
        "prompt",
        "target_accuracy",
        "schema_adherence",
        "review_sensitivity",
        "format_consistency",
    ]
].copy()


# Rename the columns so they match the terminology used in the Milestone 2 paper.
paper_results_table.columns = [
    "Prompt",
    "Target Accuracy",
    "Schema Adherence",
    "Review Sensitivity",
    "Format Consistency",
]


# Convert each numeric metric into a percentage for direct use in the paper.
for column in [
    "Target Accuracy",
    "Schema Adherence",
    "Review Sensitivity",
    "Format Consistency",
]:
    # Format each available numeric value as a percentage with one decimal place.
    paper_results_table[column] = paper_results_table[column].apply(
        lambda value: f"{value:.1%}" if pd.notna(value) else "N/A"
    )


# Display the simplified table that can be transferred directly into the Milestone 2 paper.
display(paper_results_table)

,Prompt,Target Accuracy,Schema Adherence,Review Sensitivity,Format Consistency
0,P1_Minimal,16.7%,33.3%,0.0%,100.0%
1,P2_ControlledSchema,50.0%,100.0%,0.0%,100.0%
2,P3_RoleAndCriteria,50.0%,100.0%,0.0%,100.0%
3,P4_HumanReview,83.3%,100.0%,66.7%,100.0%
4,P5_FewShot,83.3%,100.0%,66.7%,100.0%
5,P6_JSON,83.3%,100.0%,100.0%,0.0%


### Results Table for the Milestone 2 Paper

The simplified table presents the four primary evaluation dimensions used in the formal Milestone 2 paper: target accuracy, schema adherence, review sensitivity, and format consistency. These measures summarize different aspects of model behavior and should be interpreted together rather than as interchangeable indicators of performance.

The detailed notebook results provide additional context through review accuracy, unnecessary-review rate, and JSON validity. These additional measures are useful for diagnosing why a prompt succeeds or fails even though they do not all need to appear in the formal paper.

The Prompt 6 values require special interpretation because the model wrapped its JSON responses in Markdown code fences. The underlying mapping decisions and the strict JSON-format requirement should therefore be evaluated separately. The final table used in the formal paper should reflect the corrected parsing of Prompt 6 mapping decisions while retaining a strict format assessment if the output does not satisfy the instruction to return only raw valid JSON.

## 10. Inspect the Application-Oriented Prompt in Detail


In [12]:
# Filter the result table to the structured JSON experiment because it is closest to the intended application contract.
json_results = results_df[results_df["prompt"] == "P6_JSON"].copy()
# Display the source field, expected target, expected review label, and raw JSON-oriented output for manual evaluation.
display(json_results[["source_field", "expected_target", "expected_review", "model_output"]])


,source_field,expected_target,expected_review,model_output
30,patientDOB,date_of_birth,NO,"```json\n{\n ""target_field"": ""date_of_birth"",..."
31,mem_num,member_id,NO,"```json\n{\n ""target_field"": ""REVIEW"",\n ""hu..."
32,addr2,address_line_2,NO,"```json\n{\n ""target_field"": ""address_line_2""..."
33,id,REVIEW,YES,"```json\n{\n ""target_field"": ""REVIEW"",\n ""hu..."
34,name,REVIEW,YES,"```json\n{\n ""target_field"": ""REVIEW"",\n ""hu..."
35,contact,REVIEW,YES,"```json\n{\n ""target_field"": ""REVIEW"",\n ""hu..."


### Prompt Experiment Results and Interpretation

Prompt 6 is the closest approximation to the planned application interface because it requests a predictable set of fields for the target decision, human-review indicator, and rationale. Manual inspection of the six responses shows that the model followed the intended three-field structure in each case, but it enclosed the JSON objects inside Markdown code fences. As a result, the responses are visually structured as JSON but do not satisfy the strict instruction to return only raw valid JSON without additional formatting. This is an important distinction for the planned Streamlit application because an application parser should not assume that the model will always omit Markdown formatting unless that behavior is enforced or handled programmatically.

The underlying mapping decisions were substantially stronger than the strict format score alone might suggest. The model correctly selected `date_of_birth` for `patientDOB`, `address_line_2` for `addr2`, and `REVIEW` for the ambiguous `id`, `name`, and `contact` cases. The exception was `mem_num`, which should have mapped to `member_id` but was instead returned as `REVIEW`. Therefore, the main issue with Prompt 6 was not simply poor semantic mapping performance. The experiment revealed two separate concerns: one incorrect over-escalation of a straightforward field and failure to follow the strict raw-JSON formatting instruction.

This result is useful for the project because it demonstrates why application-oriented evaluation must consider both the model's semantic decision and the format of the response. A future implementation could strengthen the prompt, use structured-output controls where appropriate, or defensively remove Markdown code fences before parsing. However, the baseline result should still record the formatting deviation because the purpose of this milestone is to observe how the base model behaves under the specified prompt rather than silently correct its output.

## 11. Fine-Tuning Preparation Plan

The Milestone 2 experiments establish the baseline that will later be compared with a fully fine-tuned OpenAI model. The planned supervised fine-tuning dataset will expand to approximately 200-300 curated synthetic examples. Each example will contain the source field context and the ideal assistant response, including the approved target or `REVIEW`, a human-review indicator, and a concise rationale.

The training data should include straightforward matches, abbreviations, synonyms, datatype clues, and deliberately ambiguous cases. A held-out evaluation set should be separated before fine-tuning so the final comparison uses examples the model did not see during training. The same target accuracy, schema adherence, review sensitivity, response consistency, and qualitative rationale checks used in this notebook can then compare the base and fine-tuned models.

The exact fine-tuning base model and job parameters should be confirmed during the course's Week 7 material. The project will follow the professor's requirement to fully fine-tune rather than use LoRA.


## Conclusion

This notebook converted the Healthcare Data Mapping Assistant from a project concept into a reproducible Milestone 2 prompt-engineering experiment. The controlled target schema, synthetic non-PHI dataset, six reusable test cases, and six prompt variations made it possible to compare how different prompt-design choices affected mapping accuracy, uncertainty handling, schema adherence, and response structure.

The experiment showed that constraining the model to an approved schema improved consistency but did not by itself solve ambiguous mappings. P2_ControlledSchema and P3_RoleAndCriteria correctly handled the three straightforward cases but forced all three ambiguous cases into specific target fields. The largest improvement occurred when the model was explicitly allowed to return `REVIEW`. P4_HumanReview and P5_FewShot each achieved 83.3% target accuracy and correctly escalated two of the three ambiguous cases. These results suggest that explicit uncertainty-handling instructions are an important part of this healthcare interoperability use case.

The structured JSON experiment also demonstrated that semantic correctness and application readiness are separate concerns. Prompt 6 produced structured responses containing the requested fields, but the model wrapped the JSON in Markdown code fences rather than returning only raw JSON. Manual inspection also identified one unnecessary escalation in which `mem_num` was sent to review instead of being mapped to `member_id`. This result shows why the final application will need both reliable mapping behavior and predictable output formatting.

Overall, the Milestone 2 results provide preliminary evidence that a text-based large language model can support controlled healthcare demographic mapping, particularly for straightforward source fields. The remaining errors are concentrated in ambiguity handling, occasional over-escalation, and response-format compliance rather than a complete inability to recognize the intended healthcare concepts. These findings provide a useful baseline for the next stage of the project, which will expand the synthetic training data, fully fine-tune an approved OpenAI model, and compare the fine-tuned model with this prompt-engineered baseline on held-out examples.